# Business Metrics — SQL Reference

**Companion to:** `ds1_problem_framing.ipynb` — metric definitions, roles, and business context

**Purpose:** SQL recipes to compute every metric from the ds1 notes. One section per metric. Copy-paste ready for MySQL.

---

## §0 — Shared Schema Reference

All queries in this notebook assume the following tables. Column names are used consistently throughout.

```sql
-- users: one row per user
-- user_id        INT        unique user identifier
-- created_at     DATE       account creation date
-- channel        VARCHAR    acquisition channel (e.g. 'paid_search', 'organic', 'referral')

-- events: one row per user action
-- user_id        INT
-- event_date     DATE
-- event_type     VARCHAR    e.g. 'session', 'page_view', 'purchase', 'invite_sent', 'invite_accepted'

-- orders: one row per completed purchase
-- order_id       INT
-- user_id        INT
-- order_date     DATE
-- order_value    DECIMAL    revenue from this order

-- marketing_spend: one row per channel per day
-- spend_date     DATE
-- channel        VARCHAR
-- spend          DECIMAL    dollars spent on that channel that day

-- subscriptions: one row per active subscription per month (SaaS)
-- user_id        INT
-- month          DATE       first day of the month (e.g. '2024-01-01')
-- monthly_fee    DECIMAL    amount billed this month
-- status         VARCHAR    'active' or 'churned'

-- ad_events: one row per ad impression or click
-- user_id        INT
-- event_date     DATE
-- ad_id          VARCHAR
-- event_type     VARCHAR    'impression' or 'click'

-- transactions: one row per marketplace transaction
-- transaction_id INT
-- buyer_id       INT
-- seller_id      INT
-- transaction_date DATE
-- transaction_value DECIMAL

-- nps_responses: one row per NPS survey response
-- user_id        INT
-- response_date  DATE
-- score          INT        0–10
```

---

## §1 — DAU / WAU / MAU

**Formula:**
- DAU = COUNT(DISTINCT user_id) active on a given day
- WAU = COUNT(DISTINCT user_id) active in a given week
- MAU = COUNT(DISTINCT user_id) active in a given month

**Active** = has at least one row in `events` in the period.

### DAU — Daily Active Users

```sql
SELECT
    event_date,
    COUNT(DISTINCT user_id) AS dau
FROM events
GROUP BY event_date
ORDER BY event_date;
```

### WAU — Weekly Active Users

```sql
SELECT
    YEARWEEK(event_date, 1)          AS year_week,   -- mode 1: week starts Monday
    MIN(event_date)                  AS week_start,
    COUNT(DISTINCT user_id)          AS wau
FROM events
GROUP BY YEARWEEK(event_date, 1)
ORDER BY year_week;
```

### MAU — Monthly Active Users

```sql
SELECT
    DATE_FORMAT(event_date, '%Y-%m') AS year_month,
    COUNT(DISTINCT user_id)          AS mau
FROM events
GROUP BY DATE_FORMAT(event_date, '%Y-%m')
ORDER BY year_month;
```

**Gotchas:**
- Define "active" precisely — is a single page view enough, or does it require a meaningful action?
- Use `DISTINCT user_id` — a user opening the app 50 times in a day still counts as 1 DAU.
- `YEARWEEK(date, 1)` uses ISO week numbering (starts Monday); drop the `1` for Sunday-start weeks.

---

## §2 — DAU/MAU Ratio

**Formula:** DAU/MAU ratio = DAU ÷ MAU for the same calendar month

**Meaning:** Measures engagement depth — what fraction of monthly users return on any given day. A ratio of 0.20 means users are active ~6 days per month.

```sql
WITH
daily AS (
    SELECT
        event_date,
        DATE_FORMAT(event_date, '%Y-%m') AS year_month,
        COUNT(DISTINCT user_id)          AS dau
    FROM events
    GROUP BY event_date
),
monthly AS (
    SELECT
        DATE_FORMAT(event_date, '%Y-%m') AS year_month,
        COUNT(DISTINCT user_id)          AS mau
    FROM events
    GROUP BY DATE_FORMAT(event_date, '%Y-%m')
)
SELECT
    d.event_date,
    d.year_month,
    d.dau,
    m.mau,
    ROUND(d.dau / NULLIF(m.mau, 0), 4) AS dau_mau_ratio
FROM daily d
JOIN monthly m ON d.year_month = m.year_month
ORDER BY d.event_date;
```

**Gotchas:**
- `NULLIF(m.mau, 0)` prevents division-by-zero on months with no activity.
- DAU/MAU ratio varies by day of week — Monday vs Sunday will differ; consider a 7-day rolling average.
- A rising MAU with a falling ratio means you are acquiring users who don't return — an acquisition treadmill.

---

## §3 — D1 / D7 / D30 Retention

**Formula:**
- Dn retention = users active on day n after signup ÷ users acquired on day 0 (cohort size)

**Approach:** Self-join `events` on `user_id`, anchoring on the signup date from `users`.

```sql
WITH cohort AS (
    -- Day-0 cohort: all users and their signup date
    SELECT
        user_id,
        created_at AS cohort_date
    FROM users
),
activity AS (
    -- All event dates per user
    SELECT DISTINCT
        user_id,
        event_date
    FROM events
)
SELECT
    c.cohort_date,
    COUNT(DISTINCT c.user_id)                                                        AS cohort_size,

    -- D1: active exactly 1 day after signup
    COUNT(DISTINCT CASE
        WHEN DATEDIFF(a1.event_date, c.cohort_date) = 1 THEN c.user_id END)          AS d1_retained,
    ROUND(COUNT(DISTINCT CASE
        WHEN DATEDIFF(a1.event_date, c.cohort_date) = 1 THEN c.user_id END)
        / NULLIF(COUNT(DISTINCT c.user_id), 0), 4)                                   AS d1_retention_rate,

    -- D7
    COUNT(DISTINCT CASE
        WHEN DATEDIFF(a1.event_date, c.cohort_date) = 7 THEN c.user_id END)          AS d7_retained,
    ROUND(COUNT(DISTINCT CASE
        WHEN DATEDIFF(a1.event_date, c.cohort_date) = 7 THEN c.user_id END)
        / NULLIF(COUNT(DISTINCT c.user_id), 0), 4)                                   AS d7_retention_rate,

    -- D30
    COUNT(DISTINCT CASE
        WHEN DATEDIFF(a1.event_date, c.cohort_date) = 30 THEN c.user_id END)         AS d30_retained,
    ROUND(COUNT(DISTINCT CASE
        WHEN DATEDIFF(a1.event_date, c.cohort_date) = 30 THEN c.user_id END)
        / NULLIF(COUNT(DISTINCT c.user_id), 0), 4)                                   AS d30_retention_rate

FROM cohort c
LEFT JOIN activity a1 ON c.user_id = a1.user_id
GROUP BY c.cohort_date
ORDER BY c.cohort_date;
```

**Full retention curve (every day 0–30) per cohort month:**

```sql
WITH cohort AS (
    SELECT user_id, created_at AS cohort_date,
           DATE_FORMAT(created_at, '%Y-%m') AS cohort_month
    FROM users
),
activity AS (
    SELECT DISTINCT user_id, event_date FROM events
),
retention_raw AS (
    SELECT
        c.cohort_month,
        DATEDIFF(a.event_date, c.cohort_date)  AS day_number,
        COUNT(DISTINCT c.user_id)              AS cohort_size,
        COUNT(DISTINCT a.user_id)              AS retained
    FROM cohort c
    LEFT JOIN activity a
        ON c.user_id = a.user_id
        AND DATEDIFF(a.event_date, c.cohort_date) BETWEEN 0 AND 30
    GROUP BY c.cohort_month, DATEDIFF(a.event_date, c.cohort_date)
)
SELECT
    cohort_month,
    day_number,
    cohort_size,
    retained,
    ROUND(retained / NULLIF(cohort_size, 0), 4) AS retention_rate
FROM retention_raw
ORDER BY cohort_month, day_number;
```

**Gotchas:**
- D1 means exactly day 1, not "within 1 day" — use `DATEDIFF = 1`, not `<= 1`.
- LEFT JOIN ensures users with zero activity after signup still appear (retained = 0).
- Cohorts with fewer than 7 or 30 days of history will show NULL for future days — filter by `cohort_date <= CURDATE() - INTERVAL 30 DAY` for stable D30 numbers.

---

## §4 — K-factor

**Formula:** K-factor = (invites sent per user) × (conversion rate of invites)
- Invites sent per user = total invite_sent events ÷ total inviters
- Invite conversion rate = invite_accepted events ÷ invite_sent events

```sql
WITH invite_stats AS (
    SELECT
        COUNT(CASE WHEN event_type = 'invite_sent'     THEN 1 END) AS total_invites_sent,
        COUNT(CASE WHEN event_type = 'invite_accepted' THEN 1 END) AS total_invites_accepted,
        COUNT(DISTINCT CASE WHEN event_type = 'invite_sent' THEN user_id END) AS total_inviters
    FROM events
    WHERE event_date BETWEEN '2024-01-01' AND '2024-01-31'   -- set your measurement window
)
SELECT
    total_invites_sent,
    total_invites_accepted,
    total_inviters,
    ROUND(total_invites_sent / NULLIF(total_inviters, 0), 4)          AS invites_per_user,
    ROUND(total_invites_accepted / NULLIF(total_invites_sent, 0), 4)  AS invite_conversion_rate,
    ROUND(
        (total_invites_sent    / NULLIF(total_inviters,      0)) *
        (total_invites_accepted / NULLIF(total_invites_sent, 0))
    , 4)                                                              AS k_factor
FROM invite_stats;
```

**Gotchas:**
- K-factor > 1 means viral growth (each user brings in more than one new user on average).
- Measure over a fixed window — K-factor computed over all-time mixes cohorts and inflates the numerator.
- Ensure `invite_accepted` is attributed to the invite (not just any new signup) to avoid double-counting organic signups.

---

## §5 — NPS (Net Promoter Score)

**Formula:** NPS = % Promoters (score 9–10) − % Detractors (score 0–6)
- Passives (score 7–8) are excluded from both sides but included in the denominator.

```sql
WITH classified AS (
    SELECT
        user_id,
        score,
        CASE
            WHEN score >= 9 THEN 'promoter'
            WHEN score <= 6 THEN 'detractor'
            ELSE                 'passive'
        END AS nps_group
    FROM nps_responses
    WHERE response_date BETWEEN '2024-01-01' AND '2024-03-31'  -- measurement window
),
counts AS (
    SELECT
        COUNT(*)                                          AS total_responses,
        SUM(CASE WHEN nps_group = 'promoter'  THEN 1 ELSE 0 END) AS promoters,
        SUM(CASE WHEN nps_group = 'detractor' THEN 1 ELSE 0 END) AS detractors,
        SUM(CASE WHEN nps_group = 'passive'   THEN 1 ELSE 0 END) AS passives
    FROM classified
)
SELECT
    total_responses,
    promoters,
    detractors,
    passives,
    ROUND(promoters  / NULLIF(total_responses, 0) * 100, 1) AS pct_promoters,
    ROUND(detractors / NULLIF(total_responses, 0) * 100, 1) AS pct_detractors,
    ROUND(
        (promoters - detractors) / NULLIF(total_responses, 0) * 100
    , 1)                                                    AS nps
FROM counts;
```

**Gotchas:**
- NPS is expressed as a whole number (−100 to +100), not a fraction.
- Passives count in the denominator but not in the NPS calculation — do not drop them from the `total_responses` count.
- Use a consistent measurement window; NPS measured over different periods is not comparable.

---

## §6 — Conversion Rate

**Formula:** Conversion rate = conversions ÷ total visitors (or impressions, or sessions)

The numerator and denominator must be defined precisely — conversion rate means nothing without specifying the funnel stage.

### Simple conversion rate (visitors → purchasers)

```sql
WITH funnel AS (
    SELECT
        DATE_FORMAT(event_date, '%Y-%m') AS year_month,
        COUNT(DISTINCT CASE WHEN event_type = 'session'  THEN user_id END) AS visitors,
        COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS converters
    FROM events
    GROUP BY DATE_FORMAT(event_date, '%Y-%m')
)
SELECT
    year_month,
    visitors,
    converters,
    ROUND(converters / NULLIF(visitors, 0), 4) AS conversion_rate
FROM funnel
ORDER BY year_month;
```

### Multi-step funnel (step-by-step drop-off)

```sql
WITH steps AS (
    SELECT
        COUNT(DISTINCT CASE WHEN event_type = 'page_view'     THEN user_id END) AS step1_page_view,
        COUNT(DISTINCT CASE WHEN event_type = 'add_to_cart'   THEN user_id END) AS step2_add_to_cart,
        COUNT(DISTINCT CASE WHEN event_type = 'checkout'      THEN user_id END) AS step3_checkout,
        COUNT(DISTINCT CASE WHEN event_type = 'purchase'      THEN user_id END) AS step4_purchase
    FROM events
    WHERE event_date BETWEEN '2024-01-01' AND '2024-01-31'
)
SELECT
    step1_page_view,
    step2_add_to_cart,
    step3_checkout,
    step4_purchase,
    ROUND(step2_add_to_cart / NULLIF(step1_page_view,   0), 4) AS view_to_cart_rate,
    ROUND(step3_checkout    / NULLIF(step2_add_to_cart, 0), 4) AS cart_to_checkout_rate,
    ROUND(step4_purchase    / NULLIF(step3_checkout,    0), 4) AS checkout_to_purchase_rate,
    ROUND(step4_purchase    / NULLIF(step1_page_view,   0), 4) AS overall_conversion_rate
FROM steps;
```

**Gotchas:**
- Use `DISTINCT user_id` to count unique users per step, not total events (one user can view a page multiple times).
- The denominator changes the metric's meaning entirely — "conversion rate" without a stated denominator is ambiguous.
- Funnel steps must be ordered in time — a user who purchased without a `checkout` event suggests data quality issues.

---

## §7 — LTV (Lifetime Value)

**Formula:** LTV = avg order value × purchase frequency × avg customer lifespan

Components:
- **Avg order value** = total revenue ÷ total orders
- **Purchase frequency** = total orders ÷ total customers
- **Avg customer lifespan** = avg days between first and last order (or projected from churn rate)

```sql
WITH customer_orders AS (
    SELECT
        user_id,
        COUNT(order_id)                                   AS order_count,
        SUM(order_value)                                  AS total_revenue,
        AVG(order_value)                                  AS avg_order_value,
        MIN(order_date)                                   AS first_order_date,
        MAX(order_date)                                   AS last_order_date,
        DATEDIFF(MAX(order_date), MIN(order_date))        AS lifespan_days
    FROM orders
    GROUP BY user_id
),
aggregates AS (
    SELECT
        COUNT(user_id)                                    AS total_customers,
        SUM(order_count)                                  AS total_orders,
        AVG(avg_order_value)                              AS avg_order_value,
        AVG(order_count)                                  AS avg_purchase_frequency,
        AVG(lifespan_days) / 365.0                        AS avg_lifespan_years
    FROM customer_orders
)
SELECT
    total_customers,
    total_orders,
    ROUND(avg_order_value,        2)  AS avg_order_value,
    ROUND(avg_purchase_frequency, 2)  AS avg_purchase_frequency,
    ROUND(avg_lifespan_years,     2)  AS avg_lifespan_years,
    ROUND(avg_order_value * avg_purchase_frequency * avg_lifespan_years, 2) AS ltv
FROM aggregates;
```

**LTV by acquisition channel:**

```sql
WITH customer_orders AS (
    SELECT
        o.user_id,
        u.channel,
        COUNT(o.order_id)                           AS order_count,
        SUM(o.order_value)                          AS total_revenue,
        AVG(o.order_value)                          AS avg_order_value,
        DATEDIFF(MAX(o.order_date), MIN(o.order_date)) / 365.0 AS lifespan_years
    FROM orders o
    JOIN users u ON o.user_id = u.user_id
    GROUP BY o.user_id, u.channel
)
SELECT
    channel,
    COUNT(user_id)                                                              AS customers,
    ROUND(AVG(avg_order_value), 2)                                              AS avg_order_value,
    ROUND(AVG(order_count), 2)                                                  AS avg_frequency,
    ROUND(AVG(lifespan_years), 2)                                               AS avg_lifespan_years,
    ROUND(AVG(avg_order_value) * AVG(order_count) * AVG(lifespan_years), 2)    AS ltv
FROM customer_orders
GROUP BY channel
ORDER BY ltv DESC;
```

**Gotchas:**
- Customers with only one order have `lifespan_days = 0` — their LTV is understated. Consider a minimum lifespan assumption or a survival model.
- Averaging avg_order_value across customers (not across orders) weights each customer equally — choose deliberately.
- LTV is a projection; for new products without history, use a shorter observation window and extrapolate.

---

## §8 — CAC (Customer Acquisition Cost)

**Formula:** CAC = total acquisition spend ÷ new customers acquired (in the same period)

```sql
WITH spend AS (
    SELECT
        DATE_FORMAT(spend_date, '%Y-%m') AS year_month,
        SUM(spend)                       AS total_spend
    FROM marketing_spend
    GROUP BY DATE_FORMAT(spend_date, '%Y-%m')
),
new_customers AS (
    SELECT
        DATE_FORMAT(created_at, '%Y-%m') AS year_month,
        COUNT(user_id)                   AS new_customers
    FROM users
    GROUP BY DATE_FORMAT(created_at, '%Y-%m')
)
SELECT
    s.year_month,
    s.total_spend,
    nc.new_customers,
    ROUND(s.total_spend / NULLIF(nc.new_customers, 0), 2) AS cac
FROM spend s
JOIN new_customers nc ON s.year_month = nc.year_month
ORDER BY s.year_month;
```

**CAC by acquisition channel:**

```sql
WITH spend AS (
    SELECT
        channel,
        SUM(spend) AS total_spend
    FROM marketing_spend
    WHERE spend_date BETWEEN '2024-01-01' AND '2024-03-31'
    GROUP BY channel
),
new_customers AS (
    SELECT
        channel,
        COUNT(user_id) AS new_customers
    FROM users
    WHERE created_at BETWEEN '2024-01-01' AND '2024-03-31'
    GROUP BY channel
)
SELECT
    s.channel,
    s.total_spend,
    nc.new_customers,
    ROUND(s.total_spend / NULLIF(nc.new_customers, 0), 2) AS cac
FROM spend s
JOIN new_customers nc ON s.channel = nc.channel
ORDER BY cac ASC;
```

**Gotchas:**
- Match the time period of spend and acquired customers exactly — lag between spend and signup inflates CAC in ramp-up periods.
- CAC should never be the optimization target directly — minimizing CAC can starve growth. Use LTV/CAC instead.
- Exclude organic signups from the denominator if organic spend is not in the numerator — mixing them deflates CAC artificially.

---

## §9 — LTV/CAC Ratio

**Formula:** LTV/CAC = LTV ÷ CAC

**Benchmark:** A ratio ≥ 3× is generally considered healthy for SaaS.

```sql
WITH customer_orders AS (
    SELECT
        o.user_id,
        u.channel,
        AVG(o.order_value)                                  AS avg_order_value,
        COUNT(o.order_id)                                   AS order_count,
        DATEDIFF(MAX(o.order_date), MIN(o.order_date)) / 365.0 AS lifespan_years
    FROM orders o
    JOIN users u ON o.user_id = u.user_id
    GROUP BY o.user_id, u.channel
),
ltv_by_channel AS (
    SELECT
        channel,
        ROUND(AVG(avg_order_value) * AVG(order_count) * AVG(lifespan_years), 2) AS ltv
    FROM customer_orders
    GROUP BY channel
),
cac_by_channel AS (
    SELECT
        ms.channel,
        ROUND(SUM(ms.spend) / NULLIF(COUNT(DISTINCT u.user_id), 0), 2) AS cac
    FROM marketing_spend ms
    LEFT JOIN users u
        ON ms.channel = u.channel
        AND DATE_FORMAT(u.created_at, '%Y-%m') = DATE_FORMAT(ms.spend_date, '%Y-%m')
    GROUP BY ms.channel
)
SELECT
    l.channel,
    l.ltv,
    c.cac,
    ROUND(l.ltv / NULLIF(c.cac, 0), 2)       AS ltv_cac_ratio,
    CASE
        WHEN l.ltv / NULLIF(c.cac, 0) >= 3 THEN 'Healthy (≥3×)'
        WHEN l.ltv / NULLIF(c.cac, 0) >= 1 THEN 'Break-even (1–3×)'
        ELSE                                     'Unsustainable (<1×)'
    END                                        AS unit_economics_status
FROM ltv_by_channel l
JOIN cac_by_channel c ON l.channel = c.channel
ORDER BY ltv_cac_ratio DESC;
```

**Gotchas:**
- LTV and CAC must be computed over comparable cohorts and time windows — mixing historical LTV with current CAC distorts the ratio.
- LTV/CAC < 1 means you lose money on every customer — a structural problem, not a marketing problem.
- A high ratio can also mean under-investment in growth — use alongside MRR growth rate to check.

---

## §10 — MRR & MRR Growth Rate

**Formula:**
- MRR = SUM(monthly_fee) for all active subscribers in a month
- MRR growth rate = (MRR this month − MRR last month) ÷ MRR last month

```sql
-- MRR per month
WITH mrr_by_month AS (
    SELECT
        month,
        SUM(monthly_fee) AS mrr
    FROM subscriptions
    WHERE status = 'active'
    GROUP BY month
)
SELECT
    month,
    mrr,
    LAG(mrr) OVER (ORDER BY month)                                          AS prev_mrr,
    ROUND(
        (mrr - LAG(mrr) OVER (ORDER BY month))
        / NULLIF(LAG(mrr) OVER (ORDER BY month), 0)
    , 4)                                                                    AS mrr_growth_rate
FROM mrr_by_month
ORDER BY month;
```

**MRR broken down by movement type (new / expansion / churn):**

```sql
WITH this_month AS (
    SELECT user_id, month, monthly_fee
    FROM subscriptions
    WHERE status = 'active'
),
last_month AS (
    SELECT user_id, month, monthly_fee
    FROM subscriptions
    WHERE status = 'active'
),
movements AS (
    SELECT
        t.month,
        SUM(CASE WHEN l.user_id IS NULL                         THEN t.monthly_fee ELSE 0 END) AS new_mrr,
        SUM(CASE WHEN l.user_id IS NOT NULL
                  AND t.monthly_fee > l.monthly_fee             THEN t.monthly_fee - l.monthly_fee ELSE 0 END) AS expansion_mrr,
        SUM(CASE WHEN l.user_id IS NOT NULL
                  AND t.monthly_fee < l.monthly_fee             THEN t.monthly_fee - l.monthly_fee ELSE 0 END) AS contraction_mrr,
        SUM(CASE WHEN t.user_id IS NULL                         THEN -l.monthly_fee ELSE 0 END) AS churned_mrr
    FROM this_month t
    LEFT JOIN last_month l
        ON t.user_id = l.user_id
        AND l.month = DATE_FORMAT(DATE_SUB(t.month, INTERVAL 1 MONTH), '%Y-%m-01')
    GROUP BY t.month
)
SELECT * FROM movements ORDER BY month;
```

**Gotchas:**
- Filter `status = 'active'` before summing — churned rows in the table with a zero or NULL fee will skew MRR down.
- MRR growth rate argument order: `TIMESTAMPDIFF(MONTH, last_month, this_month)` — note the reversed order vs `DATEDIFF`.
- MRR growth without tracking churn separately is misleading — a business can show MRR growth while churning existing customers and acquiring new ones at a higher rate (acquisition treadmill).

---

## §11 — CTR (Click-Through Rate)

**Formula:** CTR = clicks ÷ impressions

```sql
-- Overall CTR by day
SELECT
    event_date,
    COUNT(CASE WHEN event_type = 'click'       THEN 1 END) AS clicks,
    COUNT(CASE WHEN event_type = 'impression'  THEN 1 END) AS impressions,
    ROUND(
        COUNT(CASE WHEN event_type = 'click'      THEN 1 END)
        / NULLIF(COUNT(CASE WHEN event_type = 'impression' THEN 1 END), 0)
    , 4)                                                   AS ctr
FROM ad_events
GROUP BY event_date
ORDER BY event_date;
```

**CTR by ad:**

```sql
SELECT
    ad_id,
    COUNT(CASE WHEN event_type = 'impression' THEN 1 END)   AS impressions,
    COUNT(CASE WHEN event_type = 'click'      THEN 1 END)   AS clicks,
    ROUND(
        COUNT(CASE WHEN event_type = 'click' THEN 1 END)
        / NULLIF(COUNT(CASE WHEN event_type = 'impression' THEN 1 END), 0)
    , 4)                                                    AS ctr
FROM ad_events
WHERE event_date BETWEEN '2024-01-01' AND '2024-01-31'
GROUP BY ad_id
ORDER BY ctr DESC;
```

**Gotchas:**
- High CTR is a vanity metric if post-click conversion is low — always pair CTR with a downstream conversion metric.
- Optimizing CTR alone incentivizes clickbait — use CTR as an operational metric, not a top-tier target.
- Unique CTR (distinct users who clicked ÷ distinct users who saw the ad) is more meaningful than raw event CTR.

---

## §12 — GMV (Gross Merchandise Value)

**Formula:** GMV = SUM(transaction_value) across all completed transactions in the period

```sql
-- GMV by month
SELECT
    DATE_FORMAT(transaction_date, '%Y-%m') AS year_month,
    COUNT(transaction_id)                  AS total_transactions,
    COUNT(DISTINCT buyer_id)               AS unique_buyers,
    ROUND(SUM(transaction_value), 2)       AS gmv,
    ROUND(AVG(transaction_value), 2)       AS avg_transaction_value
FROM transactions
GROUP BY DATE_FORMAT(transaction_date, '%Y-%m')
ORDER BY year_month;
```

**Take rate (platform revenue as % of GMV):**

```sql
-- Assumes a `platform_fees` table: transaction_id, fee_amount
WITH gmv_monthly AS (
    SELECT
        DATE_FORMAT(t.transaction_date, '%Y-%m') AS year_month,
        SUM(t.transaction_value)                 AS gmv,
        SUM(pf.fee_amount)                       AS platform_revenue
    FROM transactions t
    LEFT JOIN platform_fees pf ON t.transaction_id = pf.transaction_id
    GROUP BY DATE_FORMAT(t.transaction_date, '%Y-%m')
)
SELECT
    year_month,
    ROUND(gmv, 2)                                           AS gmv,
    ROUND(platform_revenue, 2)                              AS platform_revenue,
    ROUND(platform_revenue / NULLIF(gmv, 0) * 100, 2)      AS take_rate_pct
FROM gmv_monthly
ORDER BY year_month;
```

**Gotchas:**
- GMV is a top-line volume metric — it says nothing about platform profitability. Pair with take rate.
- Cancelled or refunded transactions should be excluded or subtracted — include only completed transactions.
- For marketplaces with both B2C and C2C transactions, segment GMV by transaction type to understand which side drives volume.

---

## Quick Reference — Metric Formulas

| Metric | Formula | Key table(s) |
|---|---|---|
| DAU / WAU / MAU | COUNT(DISTINCT user_id) per period | `events` |
| DAU/MAU ratio | DAU ÷ MAU | `events` |
| D1/D7/D30 retention | active day-n users ÷ cohort size | `users`, `events` |
| K-factor | (invites / inviters) × (accepted / invites) | `events` |
| NPS | % promoters − % detractors | `nps_responses` |
| Conversion rate | converters ÷ visitors | `events` |
| LTV | avg order value × frequency × lifespan | `orders`, `users` |
| CAC | total spend ÷ new customers | `marketing_spend`, `users` |
| LTV/CAC | LTV ÷ CAC | `orders`, `users`, `marketing_spend` |
| MRR | SUM(monthly_fee) for active subs | `subscriptions` |
| MRR growth rate | (MRR_t − MRR_{t-1}) ÷ MRR_{t-1} | `subscriptions` |
| CTR | clicks ÷ impressions | `ad_events` |
| GMV | SUM(transaction_value) | `transactions` |
